# OpenMuse en Kaggle con Persistencia Real

Este notebook permite correr OpenMuse en Kaggle y **continuar exactamente donde lo dejaste** aunque se caiga la sesión.

**Características:**
- Guardado automático cada 5 minutos
- Restauración automática al reiniciar
- 100% Kaggle (sin Google Drive ni ngrok)

**Importante:** Activá **Internet** en Settings del notebook antes de empezar.

In [ ]:
# ====================== CELDA 1: Sistema de Persistencia ======================
import sys
import os

REPO_DIR = '/kaggle/working/openmuse-kaggle-persistence'

if not os.path.exists(REPO_DIR):
    print('Clonando sistema de persistencia...')
    !git clone https://github.com/enrrutador/openmuse-kaggle-persistence.git {REPO_DIR}
else:
    print('Sistema de persistencia ya existe.')

sys.path.insert(0, REPO_DIR)

from persistence.state import OpenMuseStateManager
from persistence.auto_save import start_auto_save

manager = OpenMuseStateManager()
restored = manager.restore()

# Iniciar guardado automático cada 5 minutos
start_auto_save(manager, interval_minutes=5)

print('\n=== Sistema de persistencia listo ===')
status = manager.status()
for k, v in status.items():
    print(f'  {k}: {v}')

In [ ]:
# ====================== CELDA 2: Instalar OpenMuse ======================
print('Instalando OpenMuse (puede tardar varios minutos la primera vez)...')
!bash {REPO_DIR}/scripts/setup_openmuse.sh

In [ ]:
# ====================== CELDA 3: Configurar entorno ======================
from pathlib import Path

data_dir = str(manager.get_data_dir())

env_vars = {
    'DATA_DIR': data_dir,
    'WORKSPACE_MODE': 'sample',
    'AGENT_BACKEND': 'sample',
    'PORT': '8787',
    'HOST': '0.0.0.0',
    'PUBLIC_API_URL': 'http://localhost:8787',
    'TASK_WORKER_ENABLED': 'true',
    'WEB_SEARCH_ENABLED': 'true',
    'COMPUTER_ENABLED': 'false',
}

for key, value in env_vars.items():
    os.environ[key] = value

# Escribir .env
env_path = Path('/kaggle/working/openmuse/.env')
env_content = '\n'.join(f'{k}={v}' for k, v in env_vars.items()) + '\n'
env_path.write_text(env_content)

print('Variables de entorno configuradas:')
for k, v in env_vars.items():
    print(f'  {k} = {v}')

In [ ]:
# ====================== CELDA 4: Arrancar OpenMuse ======================
import subprocess
import time

print('Iniciando servidor de OpenMuse...')

process = subprocess.Popen(
    ['pnpm', 'dev'],
    cwd='/kaggle/working/openmuse',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ.copy()
)

print('Esperando a que el servidor arranque...')
time.sleep(12)

print('\n--- Logs del servidor ---')
lines_shown = 0
while lines_shown < 40:
    line = process.stdout.readline()
    if not line:
        break
    print(line.rstrip())
    lines_shown += 1

print('\n=== Servidor iniciado en background ===')
print('El estado se guarda automáticamente cada 5 minutos.')
print('Para forzar un guardado ahora, ejecutá la siguiente celda.')

In [ ]:
# ====================== CELDA 5: Estado y guardado manual ======================
print('=== Estado actual ===')
status = manager.status()
for k, v in status.items():
    print(f'  {k}: {v}')

print('\nForzando guardado manual...')
manager.save(reason='manual desde notebook')

print('\nListo. Podés seguir trabajando. El auto-save sigue activo.')

## Notas importantes

1. **Persistencia a largo plazo**: Hacé **Save Version** de vez en cuando para que el estado sobreviva a reinicios completos del notebook.
2. **Sin ngrok**: Todavía no hay acceso público desde el iPhone. Eso se agregará después.
3. **Si se cae**: Simplemente volvé a ejecutar las celdas desde la 1.
4. **Modo sample**: Estamos usando el modo de ejemplo de OpenMuse (no necesita claves de OpenAI ni CopilotKit).